In [3]:
import os
import pickle
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
import lightgbm as lgb

# Categorical Feature Encoding

In [4]:
import random
import polars as pl
import pickle
from pathlib import Path

DATA_DIR = Path("../../data/feature_engineered/features_daily")
ENCODER_PATH = Path("artifacts/encoders")
ENCODER_PATH.mkdir(parents=True, exist_ok=True)

categorical_cols = ["route_id", "route_type"]

# Find all daily parquet files
files = sorted(DATA_DIR.glob("*.parquet"))

# Build a small stratified sample: pick 1-2 files per month
month_files_map = {}
for file in files:
    date_str = file.stem.split("=")[-1]  # 'YYYY-MM-DD'
    month = int(date_str.split("-")[1])
    month_files_map.setdefault(month, []).append(file)

sample_files = []
for month, month_files in month_files_map.items():
    sample_files.extend(random.sample(month_files, min(len(month_files), 3)))  # pick 2 files per month

# Read selected files and extract unique categories
df_list = [pl.read_parquet(f, columns=categorical_cols) for f in sample_files]
cat_df = pl.concat(df_list).to_pandas()

# Convert to category dtype
for col in categorical_cols:
    cat_df[col] = cat_df[col].astype("category")

# Save category mappings
cat_mappings = {col: list(cat_df[col].cat.categories) for col in categorical_cols}
pickle.dump(cat_mappings, open(ENCODER_PATH / "categorical_mappings.pkl", "wb"))

print("✅ Saved stratified categorical encoders across months.")


✅ Saved stratified categorical encoders across months.


# Sampling for Hyperparameter Tuning

## Stratified Sampling Across Months

In [5]:
import random
from collections import defaultdict
import polars as pl
from pathlib import Path

def stratified_sample(files, fraction_per_month=0.05, max_files_per_month=3, seed=42):
    """
    Stratified sampling of daily parquet files for hyperparameter tuning.

    Parameters:
    - files: list of Path objects pointing to daily parquet files
    - fraction_per_month: fraction of files to sample per month (if less than max_files_per_month)
    - max_files_per_month: maximum number of files per month to sample
    - seed: for reproducibility
    """
    random.seed(seed)
    month_files = defaultdict(list)
    
    # Group files by month
    for file in files:
        date_str = file.stem.split("=")[-1]  # 'YYYY-MM-DD'
        month = int(date_str.split("-")[1])
        month_files[month].append(file)
    
    # Sample files per month
    sample_files = []
    for month, m_files in month_files.items():
        n_to_sample = min(max(1, int(len(m_files) * fraction_per_month)), max_files_per_month)
        sample_files.extend(random.sample(m_files, n_to_sample))
    
    return sorted(sample_files)

## Load Sample into Pandas with Categorical Mapping

In [6]:
import pandas as pd

def load_sample_files(files, categorical_mapping, categorical_cols):
    dfs = []
    for file in files:
        df = pl.read_parquet(file).to_pandas()
        # Apply categorical mapping
        for col in categorical_cols:
            df[col] = pd.Categorical(df[col], categories=categorical_mapping[col])
        dfs.append(df)
    sample_df = pd.concat(dfs, ignore_index=True)
    return sample_df


## Create Two Samples for Hyperparameter Tuning

In [7]:
import pickle

DATA_DIR = Path("../../data/feature_engineered/features_daily")
files = sorted(DATA_DIR.glob("*.parquet"))

categorical_cols = ["route_id", "route_type"]
cat_mapping = pickle.load(open("artifacts/encoders/categorical_mappings.pkl", "rb"))

# Coarse tuning sample (small)
coarse_files = stratified_sample(files, fraction_per_month=0.05, max_files_per_month=2)
coarse_df = load_sample_files(coarse_files, cat_mapping, categorical_cols)

# Fine tuning sample (larger)
fine_files = stratified_sample(files, fraction_per_month=0.15, max_files_per_month=5, seed=99)
fine_df = load_sample_files(fine_files, cat_mapping, categorical_cols)

print(f"Coarse sample rows: {len(coarse_df)}, Fine sample rows: {len(fine_df)}")


Coarse sample rows: 7777328, Fine sample rows: 18102346


# Two-Stage Hyperparameter Tuning

## Coarse Stage

In [ ]:
from sklearn.model_selection import GridSearchCV
import lightgbm as lgb

X_coarse = coarse_df.drop(columns=["delay", "date"])
y_coarse = coarse_df["delay"]

coarse_params = {
    "objective": ["regression"],
    "metric": ["rmse"],
    "num_leaves": [31, 50, 70],
    "learning_rate": [0.01, 0.05, 0.1],
    "feature_fraction": [0.8, 1.0],
    "max_depth": [-1, 10],
    "reg_alpha": [0, 0.1, 0.5],
    "reg_lambda": [0, 0.1, 0.5]
}

coarse_model = lgb.LGBMRegressor(n_estimators=500, n_jobs=-1)
coarse_search = GridSearchCV(
    coarse_model,
    param_grid=coarse_params,
    cv=3,
    scoring="neg_root_mean_squared_error",
    verbose=2
)
coarse_search.fit(X_coarse, y_coarse, categorical_feature=categorical_cols)

print("✅ Coarse tuning best params: ", coarse_search.best_params_)
print("✅ Coarse tuning best RMSE score: ", -coarse_search.best_score_) 

Fitting 3 folds for each of 324 candidates, totalling 972 fits
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.154106 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1808
[LightGBM] [Info] Number of data points in the train set: 5184885, number of used features: 21
[LightGBM] [Info] Start training from score 81.015032
[LightGBM] [Warning] feature_fraction is set=0.8,

## Fine Stage

In [ ]:
best = coarse_search.best_params_

fine_params = {
    "num_leaves": [best["num_leaves"] - 16, best["num_leaves"], best["num_leaves"] + 16],
    "learning_rate": [best["learning_rate"] * 0.5, best["learning_rate"], best["learning_rate"] * 1.5],
    "max_depth": [best["max_depth"] - 5, best["max_depth"], best["max_depth"] + 5],
    "feature_fraction": [0.7, 0.8, 0.9],
}

X_fine = fine_df.drop(columns=["delay", "date"])
y_fine = fine_df["delay"]

fine_model = lgb.LGBMRegressor(n_estimators=1000, n_jobs=-1)
fine_search = GridSearchCV(
    fine_model,
    param_grid=fine_params,
    cv=3,
    scoring="neg_root_mean_squared_error",
    verbose=2
)
fine_search.fit(X_fine, y_fine, categorical_feature=categorical_cols)

print("✅ Fine tuning best params:", fine_search.best_params_)


# Model Training

In [ ]:
import os
import joblib
import pickle
from pathlib import Path
import polars as pl
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ==== CONFIG ====
TRAIN_DIR = "/content/drive/MyDrive/MSc/Period 1/Introduction to Data Science/Mini Project/feature_engineered/train_daily"
TEST_DIR = "/content/drive/MyDrive/MSc/Period 1/Introduction to Data Science/Mini Project/feature_engineered/test_daily"
ENCODER_PATH = "/content/drive/MyDrive/MSc/Period 1/Introduction to Data Science/Mini Project/artifacts/encoders/categorical_mappings.pkl"
MODEL_DIR = "/content/drive/MyDrive/MSc/Period 1/Introduction to Data Science/Mini Project/artifacts/models"
MODEL_PATH = Path(MODEL_DIR) / "lightgbm_incremental.pkl"
os.makedirs(MODEL_DIR, exist_ok=True)

# ==== LOAD CATEGORY MAPPINGS ====
with open(ENCODER_PATH, "rb") as f:
    category_maps = pickle.load(f)

CATEGORICAL_FEATURES = list(category_maps.keys())
print(f"Loaded categorical mappings for: {CATEGORICAL_FEATURES}")

# ==== HELPERS ====
def read_parquet_lazy(path):
    """Reads parquet lazily using Polars and converts to Pandas for LGB."""
    return pl.scan_parquet(path).collect(engine="streaming").to_pandas()

def encode_categoricals(df, cat_maps):
    """Apply preloaded categorical mappings safely."""
    for col in CATEGORICAL_FEATURES:
        df[col] = pd.Categorical(df[col], categories=cat_maps[col])
    return df

def prepare_features(df):
    """Split features/target."""
    y = df["delay"]
    X = df.drop(columns=["delay", "date"])
    return X, y

def evaluate(model, X, y):
    """Compute regression metrics."""
    preds = model.predict(X)
    rmse = np.sqrt(mean_squared_error(y, preds))
    mae = mean_absolute_error(y, preds)
    r2 = r2_score(y, preds)
    return {"RMSE": rmse, "MAE": mae, "R2": r2}

In [ ]:
import lightgbm as lgb
import polars as pl
import pandas as pd

MODEL_PATH = Path("artifacts/models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

files = sorted(DATA_DIR.glob("*.parquet"))
cat_mappings = pickle.load(open(ENCODER_PATH / "categorical_mappings.pkl", "rb"))

categorical_cols = ["route_id", "route_type"]

params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 64,
    "max_depth": -1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1
}

model = None

for i, file in enumerate(files):
    print(f"Processing {i+1}/{len(files)}: {file.name}")
    
    df = pl.read_parquet(file).to_pandas()
    
    # Apply categorical mapping
    for col in categorical_cols:
        df[col] = pd.Categorical(df[col], categories=cat_mappings[col])
    
    X = df.drop(columns=["delay", "date"])
    y = df["delay"]
    
    train_data = lgb.Dataset(X, label=y, categorical_feature=categorical_cols)
    
    if model is None:
        model = lgb.train(
            params=params,
            train_set=train_data,
            num_boost_round=500
        )
    else:
        model = lgb.train(
            params=params,
            train_set=train_data,
            num_boost_round=500,
            init_model=model
        )

# Save final model
model.save_model(MODEL_PATH / "lgb_transport_delay_incremental.txt")
print("✅ Final incremental model saved.")


# Evaluation

In [ ]:
# ==== LOAD TRAINED MODEL ====
model = joblib.load(MODEL_PATH)

test_files = sorted(Path(TEST_DIR).rglob("date=*.parquet"))
metrics_all = []

for file in test_files:
    df_test = read_parquet_lazy(file)
    df_test = encode_categoricals(df_test, category_maps)
    X_test, y_test = prepare_features(df_test)

    metrics = evaluate(model, X_test, y_test)
    metrics["file"] = file.name
    metrics_all.append(metrics)
    print(f"✅ Evaluated {file.name}: RMSE={metrics['RMSE']:.3f}")
    print(f"✅ Evaluated {file.name}: MAE={metrics['MAE']:.3f}")
    print(f"✅ Evaluated {file.name}: R2={metrics['R2']:.3f}")

# ==== SAVE SUMMARY ====
metrics_df = pd.DataFrame(metrics_all)
metrics_df.to_csv(Path(MODEL_DIR) / "test_performance_summary.csv", index=False)
print("✅ Test evaluation completed and saved.")